In [8]:
# input
rep_clique_file = "./tmp/repId-cliques.tsv"
aligned_result_file = "../../data/pred_cluster_aligned_result.tsv"
# output
clique_2_divergence = "./data/clique_2_diverged.tsv"

In [2]:
import pandas as pd


dfs = []
for i in range(12):
    dfs.append(pd.read_table(f"{rep_clique_file}.part_{i}"))
df = pd.concat(dfs)
df['cliques'] = df['cliques'].map(lambda x: [i.split(",") for i in x.split(";")])
df['num_clique'] = df['cliques'].map(lambda x: len(x))
df_aligned = pd.read_table(aligned_result_file)

In [3]:
import numpy as np

np.histogram(df['num_clique'], bins=range(1, 10))

(array([682563,  76787,   8480,   1976,    626,    263,    156,    117]),
 array([1, 2, 3, 4, 5, 6, 7, 8, 9]))

## 2 clique case

In [4]:
df = df[df['num_clique'] == 2]
target_reps = set(df['rep_id'])
df_aligned = df_aligned[df_aligned['rep_id'].map(lambda x: x in target_reps)]

In [5]:
seq_id_to_pred_posi = dict(zip(df_aligned['seq_id'], df_aligned['is_pred'].map(lambda x: set([index for index, c in enumerate(x.split(",")) if c == "1"]))))
seq_id_to_aligned_resi = dict(zip(df_aligned['seq_id'], df_aligned['real_aa'].map(lambda x: x.split(","))))

def get_pred_posis_for_clique(c):
    result = set()
    for i in c:
        result |= seq_id_to_pred_posi[i]

    return result

def has_pred_intersection_in_two_cliques(c1, c2):
    preds_1 = get_pred_posis_for_clique(c1)
    preds_2 = get_pred_posis_for_clique(c2)

    return len(preds_1 & preds_2) != 0

def get_posi2resi_for_clique(preds: set, c: list):
    posi2resi = dict()
    for posi in preds:
        residues = set()
        for seq_id in c:
            aligned_aas = seq_id_to_aligned_resi[seq_id]
            residues.add(aligned_aas[posi])
        posi2resi[posi] = residues
    return posi2resi

def has_matched_resi_in_preds(pred_to_resi_1, pred_to_resi_2):

    has_matched_in_preds = False
    for p in pred_to_resi_1.keys():
        clique_1_resi = pred_to_resi_1[p]
        clique_2_resi = pred_to_resi_2[p]
        if len(clique_1_resi & clique_2_resi) != 0:
            has_matched_in_preds = True
            break

    return has_matched_in_preds

def has_matched_resi_in_two_cliques(c1, c2):
    preds_1 = get_pred_posis_for_clique(c1)
    preds_2 = get_pred_posis_for_clique(c2)

    preds_1_to_clique_1_resi = get_posi2resi_for_clique(preds_1, c1)
    preds_1_to_clique_2_resi = get_posi2resi_for_clique(preds_1, c2)
    preds_2_to_clique_1_resi = get_posi2resi_for_clique(preds_2, c1)
    preds_2_to_clique_2_resi = get_posi2resi_for_clique(preds_2, c2)

    has_matched_in_preds_1 = has_matched_resi_in_preds(preds_1_to_clique_1_resi, preds_1_to_clique_2_resi)
    has_matched_in_preds_2 = has_matched_resi_in_preds(preds_2_to_clique_1_resi, preds_2_to_clique_2_resi)

    if (not has_matched_in_preds_1) and (not has_matched_in_preds_2):
        return False
    else: return True


df['has_intersection'] = df['cliques'].map(lambda x: has_pred_intersection_in_two_cliques(x[0], x[1]))
df['has_matched_resi'] = df['cliques'].map(lambda x: has_matched_resi_in_two_cliques(x[0], x[1]))
len(df)
df = df[df['has_intersection'] == False]
len(df)
df = df[df['has_matched_resi'] == False]
len(df)

76787

8640

1006

In [10]:
df['cliques'] = df['cliques'].map(lambda x: ";".join([",".join(y) for y in x]))
df.to_csv(clique_2_divergence, sep="\t", index=None, columns=["rep_id", "cliques"])